# Calibrating HBV hydrological model using SCE run locally outside container forced with ERA5 forcing data
In this notebook we will demonstrate how to calibrate the HBV model with Shuffled Complex Evolution (SCE) and works as an example for calibration methods that use a function that needs to be passed to an optimization algorithm. 

Instead of an ensemble of models, we will need to construct a function that does the entire model run.

We will use two new packages not used before:

- the sceua package implements the optimisation algorithm. Infortunately we run into a dependency problem: ESMValTool, which is part of eWaterCycle, requires numpy 1.26.4, sceua requires numpy >= 2. So we install, and subsequently de-install sceua in this notebook.
- the hydrobm package implements common metrics used in hydrology, including the kge, which we use here.  

This assumes you have already seen [the notebook on how to do Monte Carlo calibration in eWaterCycle](step_2a_calibrate_HBV_montecarlo.ipynb)

In [1]:
# General python
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import json
import os

# Niceties
from rich import print
from tqdm import tqdm

In [2]:
# General eWaterCycle
import ewatercycle
import ewatercycle.models
import ewatercycle.forcing

In [3]:
"""On Spider we want to not change our environment every run, so we have to do the pip installing separately.
This means that that is already done and handled by papermill, but since we still want this workflow to work on a SRC machine we need to do the pip installing"""
if "ewater" in str(Path.home()):
    # we are on spider
    run_pips = False
else:
    run_pips = True

In [4]:
%%capture
if run_pips:
    # We need the sceua package. If that is not available on your machine,
    # uncomment the line below to install it

    !pip install sceua

In [5]:
import sceua

In [6]:
if run_pips:
    # We make use of the hydrobm package to calculate metrics
    !pip install hydrobm

Defaulting to user installation because normal site-packages is not writeable


In [7]:
from hydrobm.metrics import calculate_metric

In [8]:
# Parameters, these get changed when running on HPC
# country = "australia"
# region_id = "camelsaus_102101A"
settings_path = "settings.json"

In [9]:
# Load settings
# Read from the JSON file
with open(settings_path, "r") as json_file:
    settings = json.load(json_file)

In [10]:
display(settings)

{'caravan_id': 'camelsaus_102101A',
 'country': 'australia',
 'calibration_start_date': '1994-08-01T00:00:00Z',
 'calibration_end_date': '2004-07-31T00:00:00Z',
 'validation_start_date': '2004-08-01T00:00:00Z',
 'validation_end_date': '2014-07-31T00:00:00Z',
 'future_start_date': '2029-08-01T00:00:00Z',
 'future_end_date': '2049-08-31T00:00:00Z',
 'CMIP_info': {'dataset': ['MPI-ESM1-2-LR'],
  'ensembles': ['r1i1p1f1', 'r2i1p1f1', 'r3i1p1f1'],
  'experiments': ['historical', 'ssp126', 'ssp245', 'ssp370', 'ssp585'],
  'project': 'CMIP6',
  'grid': 'gn'},
 'koppen_raster_path': '/data/shared/climate-data/koppen_geiger/1991_2020/koppen_geiger_0p00833333.tif',
 'base_path': '/home/mmelotto/ewatercycleClimateImpact/HBV',
 'path_caravan': '/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/caravan',
 'path_ERA5': '/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/ERA5',
 'path_DestinE': '/home/mmelotto/ewatercycleClimateImpa

In [11]:
# We check if we already ran this test
file_parameters_path = settings['path_output'] + "/" + settings['caravan_id'] + "_params_SCE.csv"
need_to_run = True

if os.path.exists(file_parameters_path):
    display(f"File already exists: {file_parameters_path}")
    need_to_run = False

# need_to_run = True

'File already exists: /home/mmelotto/ewatercycleClimateImpact/HBV/output_data/australia/camelsaus_102101A/camelsaus_102101A_params_SCE.csv'

# Pre-generated observations of discharge from caravan
Here we re-load the disharge data we generated in [this](step_0a_select_caravan_region_time_and_scenarios.ipynb) notebook.

In [12]:
# Load the caravan forcing object
caravan_data_object = ewatercycle.forcing.sources['CaravanForcing'].load(directory=settings['path_caravan'])
display(caravan_data_object)

CaravanForcing(start_time='1994-08-01T00:00:00Z', end_time='2014-07-31T00:00:00Z', directory=PosixPath('/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/caravan'), shape=PosixPath('/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/caravan/camelsaus_102101A.shp'), filenames={'tasmin': 'camelsaus_102101A_1994-08-01_2014-07-31_tasmin.nc', 'tas': 'camelsaus_102101A_1994-08-01_2014-07-31_tas.nc', 'tasmax': 'camelsaus_102101A_1994-08-01_2014-07-31_tasmax.nc', 'Q': 'camelsaus_102101A_1994-08-01_2014-07-31_Q.nc', 'pr': 'camelsaus_102101A_1994-08-01_2014-07-31_pr.nc', 'evspsblpot': 'camelsaus_102101A_1994-08-01_2014-07-31_evspsblpot.nc'})

## Pre-generated ERA5 forcing data for HBV model
Here we load the ERA5 data we generated in [this](step_1a_generate_historical_forcing) notebook

In [13]:
load_location = Path(settings['path_ERA5']) / "work" / "diagnostic" / "script" 
ERA5_forcing_object = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

In [14]:
display(ERA5_forcing_object)

LumpedMakkinkForcing(start_time='1994-08-01T00:00:00Z', end_time='2014-07-31T00:00:00Z', directory=PosixPath('/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/ERA5/work/diagnostic/script'), shape=PosixPath('/home/mmelotto/ewatercycleClimateImpact/HBV/forcing_data/australia/camelsaus_102101A/ERA5/work/diagnostic/script/camelsaus_102101A.shp'), filenames={'pr': 'OBS6_ERA5_reanaly_1_day_pr_1994-2014.nc', 'tas': 'OBS6_ERA5_reanaly_1_day_tas_1994-2014.nc', 'rsds': 'OBS6_ERA5_reanaly_1_day_rsds_1994-2014.nc', 'evspsblpot': 'Derived_Makkink_evspsblpot.nc'})

## Calibration basics and objective function
In model calibration, we are looking for a set of parameters such that when the model is run with that set of parameters, we get the best model output. What "best" means differs per application or research question. In general, we like some model outputs to be as close as possible to observations. For this purpose we create an objective function that takes the model output of interest and observations as inputs and calculates some score that shows goodness of fit. Here we use a RMS difference function:

In [15]:
def calibrationObjective(model_output, observation, start_calibration, end_calibration, metric_name = 'rmse'):
    '''A function that takes in two dataFrames, interpolates the model output to the
    observations and calculates the average absolute difference between the two. '''

    # Combine the two in one dataFrame and interpolate, to make sure times match
    hydro_data = pd.concat([model_output.reindex(observation.index, method = 'ffill'), observation], axis=1,
                           keys=['model','observation'])

    # Only select the calibration period
    hydro_data = hydro_data[hydro_data.index > pd.to_datetime(pd.Timestamp(start_calibration).date())]
    hydro_data = hydro_data[hydro_data.index < pd.to_datetime(pd.Timestamp(end_calibration).date())]

    obs = hydro_data['observation'].to_numpy()
    sim = hydro_data['model'].to_numpy()

    metric_value  = calculate_metric(obs,sim,metric_name)

    return metric_value

    # match metric:
    #     case 'rmse':
    #         # Calculate mean absolute difference
    #         squareDiff = (sim - obs)**2
    #         rmse = np.sqrt(np.mean(squareDiff))
    #         return rmse
    #     case 'nse':
    #         # Calculate Nash Shutcliff Efficiency
    #         nse = 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)
    #         return nse
    #     case 'kge':
    #         # Calculate Kling Gupta Efficiency
    #         if (np.std(obs) == 0) or (np.std(sim) == 0):
    #             r = 0
    #         else:
    #             r = np.corrcoef(obs, sim)[0, 1]
    #         alpha = np.std(sim) / np.std(obs)
    #         beta = np.mean(sim) / np.mean(obs)
    #         kge = 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    #         return kge


In [16]:
def objective_function_sorted_maximum(model_output, observation, start_calibration, end_calibration):
    calibration_start_time = pd.to_datetime(start_calibration)
    calibration_end_time = pd.to_datetime(end_calibration)

    model_output.index = model_output.index.tz_localize(None)  # Remove timezone
    observation.index = observation.index.tz_localize(None)
    start = calibration_start_time.tz_localize(None)
    end = calibration_end_time.tz_localize(None)

    model_aligned = model_output.reindex(observation.index, method='ffill')
    df = pd.concat({'model': model_aligned, 'observation': observation}, axis=1)
    df = df.loc[start:end].dropna()

    sim = df['model'].to_numpy()
    obs = df['observation'].to_numpy()

    # Sort data from high to low
    sorted_model_data = np.sort(sim)[::-1] 
    sorted_obs_data = np.sort(obs)[::-1]

    # Calculate return periods
    m = len(sorted_model_data)
    rank = np.arange(1, m + 1)
    return_periods_days_model = (m + 1) / rank
    # return_periods_years_model = return_periods_days_model / 365.25


    m = len(sorted_obs_data)
    rank = np.arange(1, m + 1)
    return_periods_days_obs = (m + 1) / rank
    # return_periods_years_obs = return_periods_days_obs / 365.25

    sorted_model_data_subset = sorted_model_data
    sorted_obs_data_subset = sorted_obs_data


    if len(sorted_model_data_subset) != len(sorted_obs_data_subset):
        print(len(sorted_model_data_subset), len(sorted_obs_data_subset))
        raise ValueError("observation and data not equal length")

    objective_this_model = np.sqrt(np.mean((sorted_model_data_subset - sorted_obs_data_subset)**2))

    if np.isnan(objective_this_model):
        return np.inf

    return objective_this_model

## Create a function that runs the entire model and evaluates it


In [17]:
# Create an array with parameter values.

# First set minimum and maximum values on the parameters
p_min_initial = np.array([0,   0.2,  40,    .5,   .001,   1,     .01,  .0001,   0.01])
p_max_initial = np.array([8,    1,  800,   4,    .3,     10,    .1,   .01,   10.0])

p_initial = (p_min_initial + p_min_initial)/2

In [18]:
# Print parameter names and values for first ensemble member
param_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
display(list(zip(param_names, np.round(p_initial, decimals=3))))

[('Imax', np.float64(0.0)),
 ('Ce', np.float64(0.2)),
 ('Sumax', np.float64(40.0)),
 ('Beta', np.float64(0.5)),
 ('Pmax', np.float64(0.001)),
 ('Tlag', np.float64(1.0)),
 ('Kf', np.float64(0.01)),
 ('Ks', np.float64(0.0)),
 ('FM', np.float64(0.01))]

In [19]:
# Set initial state values
#               Si,  Su, Sf, Ss, Sp
s_0 = np.array([0,  100,  0,  5,  0])

In [20]:
# Create a dataframe for the observations
ds_observation = xr.open_mfdataset([caravan_data_object['Q']]).to_pandas()
ds_observation = ds_observation['Q']

In [21]:
def runModel(params, forcing_object, observations, initial_state, start_calibration, end_calibration):
    # Create model object, notice the forcing object.
    model = ewatercycle.models.HBVLocal(forcing=forcing_object)

    # Create config file in model.setup()
    config_file, _ = model.setup(parameters=params, initial_storage=initial_state,
                                start_time = start_calibration, end_time = end_calibration, cfg_dir=settings["path_output"])
    # Initialize model
    model.initialize(config_file)
    # Run model, capture calculated discharge and timestamps
    Q_m = []
    time = []
    while model.time < model.end_time:
        model.update()
        Q_m.append(model.get_value("Q")[0])
        time.append(pd.Timestamp(model.time_as_datetime))
    # Finalize model (shuts down container, frees memory)
    model.finalize()

    # Make a pandas series
    model_output = pd.Series(data=Q_m, name="modelled discharge", index=time)

    return model_output

In [22]:
def runModelReturnObjective(params, forcing_object, observations, initial_state, start_calibration, end_calibration):

    model_output = runModel(params, forcing_object, observations, initial_state, start_calibration, end_calibration)

    kge = calibrationObjective(model_output, observations, start_calibration, end_calibration, metric_name = 'kge')
    nse = calibrationObjective(model_output, observations, start_calibration, end_calibration, metric_name = 'nse')
    sorted_max = objective_function_sorted_maximum(model_output, observations, start_calibration, end_calibration)
    
    # kge is best if high, so we do one minus kge to make sure that sce optimizes correctly
    # since sce always minimizes

    score = np.sqrt( (1 - kge)**2 + (0 - sorted_max)**2 + (1 - nse)**2)

    return score

## Find best parameter set
By calculating the objective function for each model output, we can search the combination of parameters with the lowest objective function.

In [23]:
# Define parameter bounds as a sequence of (min, max) pairs
bounds = [(p_min_initial[n], p_max_initial[n]) for n in range(len(p_min_initial))]

display(bounds)



[(np.float64(0.0), np.float64(8.0)),
 (np.float64(0.2), np.float64(1.0)),
 (np.float64(40.0), np.float64(800.0)),
 (np.float64(0.5), np.float64(4.0)),
 (np.float64(0.001), np.float64(0.3)),
 (np.float64(1.0), np.float64(10.0)),
 (np.float64(0.01), np.float64(0.1)),
 (np.float64(0.0001), np.float64(0.01)),
 (np.float64(0.01), np.float64(10.0))]

In [ ]:
if need_to_run:
    # Run optimization
    result = sceua.minimize(runModelReturnObjective, bounds, args=(ERA5_forcing_object, ds_observation, s_0, settings['calibration_start_date'],
                                                                  settings['calibration_end_date']), seed=42, max_workers=1)


In [ ]:
if need_to_run:
    # Access the optimization results
    best_params = result.x
    best_function_value = result.fun
    num_iterations = result.nit
    num_function_evaluations = result.nfev

In [ ]:
if need_to_run:
    display(best_function_value)

In [ ]:
if need_to_run:
    model_output = runModel(best_params, ERA5_forcing_object, ds_observation, s_0, 
                                              settings['calibration_start_date'], settings['calibration_end_date'])

In [ ]:
if need_to_run:
    # Calculate NSE for the calibration period using the best parameters
    best_nse = calibrationObjective(model_output, ds_observation,
                                    settings['calibration_start_date'], settings['calibration_end_date'],
                                    metric_name='nse')
    display(f"Calibration NSE: {best_nse}")

    best_kge = calibrationObjective(model_output, ds_observation,
                                    settings['calibration_start_date'], settings['calibration_end_date'],
                                    metric_name='kge')
    display(f"Calibration KGE: {best_kge}")

    best_sorted_max = objective_function_sorted_maximum(model_output, ds_observation, settings['calibration_start_date'], settings['calibration_end_date'])

In [ ]:
if need_to_run:
    # Create a dataframe for the observations
    ds_observation = xr.open_mfdataset([caravan_data_object['Q']]).to_pandas()

In [ ]:
if need_to_run:
    # Zoom to 2 years starting from 2000-08-01
    start_time = pd.Timestamp('2000-08-01')
    zoom_end = start_time + pd.DateOffset(years=2)
    
    obs_zoom = ds_observation["Q"].loc[start_time:zoom_end]
    model_zoom = model_output.loc[start_time:zoom_end]
    
    # Make a plot of the model output
    obs_zoom.plot(label='observed discharge')
    model_zoom.plot(lw=2.5, label='modelled discharge')
    plt.title(f"Calibrated HBV model (2000-2002)\nNelder-Mead: {best_function_value:.3f} KGE: {best_kge:.3f}, NSE: {best_nse:.3f}")
    plt.xlabel('Time')
    plt.ylabel("Discharge (mm/d)")
    plt.legend()
    plt.savefig(settings["figure_output"] + "/calibration.png")
    plt.show()

## Save results
We want to save these results to file to be able to load them in other studies.

In [ ]:
if need_to_run:
    # Save to csv file
    np.savetxt(Path(settings["path_output"]) / (settings['caravan_id'] + "_params_SCE.csv"), best_params, delimiter=",")

In [ ]:
if need_to_run:
    # Save calibration KGE and NSE to results.json (same directory as settings.json)
    results_path = Path(settings["results_path"])
    results = json.loads(results_path.read_text()) if results_path.exists() else {}

    # results["caravan_id"] = settings["caravan_id"]
    results["calibration_HBV"] = {
        "Distance Metric": round(best_params, 4)
        "Nelder-Mead": round(best_sorted_max, 4),  # should be close to 0
        "KGE": round(best_kge, 4),  # best if close to 1
        "NSE": round(best_nse, 4),  # best if close to 1
        "period": f"{settings['calibration_start_date'][:10]} / {settings['calibration_end_date'][:10]}",
    }

    results_path.write_text(json.dumps(results, indent=4))
    print(f"Calibration Nelder-Mead = {results['calibration_HBV']['Nelder-Mead']}, Calibration KGE = {results['calibration_HBV']['KGE']}, NSE = {results['calibration_HBV']['NSE']} written to {results_path}")

In [ ]:
# Remove all temporary directories made by the optimization algo.

!rm -rf hbvlocal*

In [ ]:
%%capture
if run_pips:
    # Re-install correct numpy for ESMVAlTool
    !pip install esmvaltool